# 01 — Data Loading & Cleaning
## Vehicle Telematics Project

**What this notebook does:**
- Loads raw `allcars.csv` (3.1M rows, 16 vehicles)
- Forces numeric types — raw file contains repeated header rows due to dataset concatenation
- Applies sensor range filters to remove glitches
- Flags vehicles with missing or unreliable KPL sensors
- Exports cleaned dataset to `data/processed/clean.parquet`

**Author:** Paşan Sancak · [LinkedIn](https://linkedin.com/in/pasansancak) · [GitHub](https://github.com/pasansancak)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
sys.path.append('..')
from src.features import load_and_clean, get_excluded_devices
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

plt.rcParams['font.family'] = 'monospace'
sns.set_theme(style='darkgrid', palette='muted')
print('Setup complete ✓')

Setup complete ✓


## 1. Load Raw Data

In [2]:
raw = pd.read_csv('../data/allcars.csv', low_memory=False)

print(f'Raw shape  : {raw.shape[0]:,} rows x {raw.shape[1]} columns')
print(f'Columns    : {list(raw.columns)}')
print()
raw.head(3)

Raw shape  : 3,120,272 rows x 17 columns
Columns    : ['tripID', 'deviceID', 'timeStamp', 'accData', 'gps_speed', 'battery', 'cTemp', 'dtc', 'eLoad', 'iat', 'imap', 'kpl', 'maf', 'rpm', 'speed', 'tAdv', 'tPos']



,tripID,deviceID,timeStamp,accData,gps_speed,battery,cTemp,dtc,eLoad,iat,imap,kpl,maf,rpm,speed,tAdv,tPos
0,1,0.0,2017-12-22 18:43:05,10c0f8e00448fa18c80515d30000000000000000000000...,24.2612,0.0,66.0,0.0,28.6275,40.0,97.0,0.0,0.0,1010.75,23.0,0.0,0.0
1,1,0.0,2017-12-22 18:43:06,1138f8c804780a1ebdf718bcf919d10617c8e301b31017...,23.15,0.0,66.0,0.0,33.7255,40.0,98.0,0.0,0.0,815.5,21.0,0.0,0.0
2,1,0.0,2017-12-22 18:43:07,10f0f89804480612c30010c30714ce0520b7f41dbdf118...,18.7052,0.0,66.0,0.0,43.1373,40.0,98.0,0.0,0.0,862.25,17.0,0.0,0.0


## 2. Clean

The raw file contains repeated header rows embedded in the data (artifact of dataset concatenation).
`pd.to_numeric(errors='coerce')` converts these to NaN, which are then dropped.

In [3]:
raw_len = len(raw)
df      = load_and_clean('../data/allcars.csv')

print(f'Raw rows     : {raw_len:,}')
print(f'Clean rows   : {len(df):,}')
print(f'Removed      : {raw_len - len(df):,} ({(raw_len-len(df))/raw_len*100:.1f}%)')
print()
print(f'Devices      : {sorted(df["deviceID"].dropna().unique().astype(int).tolist())}')
print(f'Trips        : {df["tripID"].nunique()}')
print(f'Date range   : {df["timeStamp"].min().date()} → {df["timeStamp"].max().date()}')
print()
df[['speed','rpm','eLoad','kpl','cTemp','tPos']].describe().round(2)

Raw rows     : 3,120,272
Clean rows   : 3,021,632
Removed      : 98,640 (3.2%)

Devices      : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16]
Trips        : 431
Date range   : 2017-11-18 → 2018-01-31



,speed,rpm,eLoad,kpl,cTemp,tPos
count,3021632.00,3021632.00,3021632.00,3021632.00,3021599.00,3021632.00
mean,22.06,947.72,31.24,4.23,60.31,12.35
std,25.32,666.90,28.02,5.22,34.81,25.52
min,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.00,0.00,0.00,0.00,39.00,0.00
50%,14.00,972.00,27.84,2.19,80.00,0.00
75%,35.00,1400.00,50.98,7.08,89.00,14.90
max,149.00,5472.00,99.61,25.00,100.00,99.61


## 3. KPL Sensor Quality Check

In [4]:
EXCLUDED = get_excluded_devices(df)

kpl_summary = df.groupby('deviceID').agg(
    kpl_sum    = ('kpl', 'sum'),
    kpl_mean   = ('kpl', 'mean'),
    row_count  = ('kpl', 'count')
).round(2)
kpl_summary['excluded'] = kpl_summary.index.isin(EXCLUDED)
kpl_summary.index = kpl_summary.index.astype(int)

print('KPL sensor availability per vehicle:')
print(kpl_summary.to_string())
print()
print(f'Excluded from fuel analyses: {[int(d) for d in EXCLUDED]}')
print('  → Vehicles 0,1,2,4,11 : KPL sensor not recording')
print('  → Vehicle 7            : KPL sensor intermittent/unreliable')
print('  → Vehicle 9            : Valid but anomalously high KPL — likely different engine')

KPL sensor availability per vehicle:
             kpl_sum  kpl_mean  row_count  excluded
deviceID                                           
0               0.00      0.00     290869      True
1               0.00      0.00      10280      True
2               0.00      0.00      27166      True
3          889846.97      6.03     147541     False
4               0.00      0.00       9880      True
5         1115421.84      4.11     271612     False
6           41494.92      1.95      21271     False
7           82002.55      0.45     180364      True
8          153634.95      3.55      43217     False
9         2502848.96      6.93     361422     False
10        4525149.99      6.06     746820     False
11              0.00      0.00       7929      True
12        2889036.02      3.74     773411     False
14              0.00      0.00       1962      True
16         571336.56      4.47     127888     False

Excluded from fuel analyses: [0, 1, 2, 4, 11, 14, 7]
  → Vehicles 0,1,2,4,11 :

## 4. Missing Values Summary

In [5]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

print('Missing values in clean dataset:')
for col in missing.index:
    print(f'  {col:<12} {missing[col]:>8,}  ({missing_pct[col]:.2f}%)')

Missing values in clean dataset:
  cTemp              33  (0.00%)


## 5. Export Cleaned Dataset

In [6]:
# Export as parquet — much faster to load in subsequent notebooks
df.to_parquet('../data/processed/clean.parquet', index=False)

# Also export excluded device list
import json
with open('../data/processed/excluded_devices.json', 'w') as f:
    json.dump([float(d) for d in EXCLUDED], f)

print(f'Exported clean.parquet  : {len(df):,} rows')
print(f'Exported excluded_devices.json : {[int(d) for d in EXCLUDED]}')

Exported clean.parquet  : 3,021,632 rows
Exported excluded_devices.json : [0, 1, 2, 4, 11, 14, 7]
